# MOD-12930 — Fair FT.HYBRID Benchmark

Implements [`../MOD-12930-benchmark-plan.md`](../MOD-12930-benchmark-plan.md).
All machinery (server management, query generation, gates, timing) lives in
[`bench_lib.py`](bench_lib.py) — this notebook is the narrative and the results.

**Question**: PERF-473 saw FT.HYBRID degrade ~6× moving from 10K to 100K docs while an
FT.AGGREGATE "equivalent" stayed flat. Is that an implementation problem, or inherent to
what FT.HYBRID computes?

**Timed contenders** (all server-side; BM25STD; final output fixed at top-10):

| contender | what it is |
|---|---|
| `hybrid_linear` / `hybrid_rrf` | `FT.HYBRID` under test |
| `search_branch` | its text branch standalone: `FT.SEARCH … WITHSCORES LIMIT {W-10} 10` — native score sort, top-W heap, 10-row reply (an `ADDSCORES+SORTBY` aggregate is ~30-45% slower for the same job and would overstate branch cost) |
| `vsim_branch` | its vector branch standalone: `*=>[KNN {K} …]` aggregate, K-sized sort, 10-row reply |

(The PERF-473 rerank aggregate was dropped from the matrix — its story is settled in earlier
runs: flat scaling, ≤42% overlap with true fusion, different semantics.)

**Matrix**: size (10K / 100K / 500K) × WORKERS (0 / 6) × text selectivity (selective /
medium / broad — *measured* via `LIMIT 0 0`, never assumed; broad is the PERF-473-style
`~@text:(17 tokens)` that matches 100% of the corpus).

**Fields axis** — every contender runs in two modes, never mixed:
- `none`: top-10 keys+scores only (`__key` is returned by FT.HYBRID natively). Work is
  identical across contenders — the exact-decomposition mode where ε and the accounting
  identity are meaningful.
- `title+text`: same commands + document fields. Strict work-equality is impossible here
  (the engine loads WINDOW+K rows per hybrid branch *pre-fusion*, while a pull-based
  standalone query only loads what it returns), so this mode compares **same goal, own
  cost** — and the per-contender delta between modes measures hybrid's load amplification.

**Retrieval depth scales with selectivity** — K/WINDOW = 10/20 (selective), 100/200
(medium), 1000/2000 (broad) — so the vector subquery and the merger are exercised
proportionally instead of staying at constant trivial cost while the text branch scales
(K = WINDOW/2, respecting the engine's KNN K ≤ WINDOW constraint; depth is constant
across dataset sizes so degradation ratios stay interpretable).

**Gates**: before timing, FT.HYBRID must return exactly what an untimed two-query oracle
fusion returns (tie-aware score comparison). A failed gate voids that configuration.

In [1]:
import json
import pandas as pd
import bench_lib as B

titles, texts, emb, corpus_max = B.load_data()
print('module:', B.MODULE)
print(f'{emb.shape[0]} rows, dim={emb.shape[1]}, corpus={corpus_max} + {emb.shape[0]-corpus_max} held-out query rows')

module: /Users/itzik.vaknin/dev/RediSearchClean/.claude/worktrees/warm-swimming-stallman/bin/macos-arm64v8-release/search-community/redisearch.so
500256 rows, dim=512, corpus=500000 + 256 held-out query rows


## Run the full matrix

Per dataset size: load + index, generate the query set, run the gates, then time every workers × selectivity × contender cell (3000 reps over 256 distinct queries, 300 warm-up).

In [2]:
results, gates, profiles, meta = B.run_matrix(titles, texts, emb, corpus_max)

with open('results.json', 'w') as f:
    json.dump(dict(meta=meta, results=results, gates=gates, profiles=profiles), f,
              indent=2, default=str)
pd.DataFrame(results).to_csv('results.csv', index=False)
print('saved results.json / results.csv')


===== dataset size 10000 =====


loaded+indexed 10000 docs in 2.8s (hash_indexing_failures=0)


selective  |matches| mean=3 median=1 min=0 max=101  (N=10000)
medium     |matches| mean=1055 median=1031 min=386 max=2651  (N=10000)
broad      |matches| mean=10000 median=10000 min=10000 max=10000  (N=10000)
{'size': 10000, 'selectivity': 'selective', 'matches_mean': 2.80078125, 'gate_linear': '16/16', 'gate_rrf': '16/16'}
{'size': 10000, 'selectivity': 'medium', 'matches_mean': 1055.02734375, 'gate_linear': '16/16', 'gate_rrf': '16/16'}


{'size': 10000, 'selectivity': 'broad', 'matches_mean': 10000.0, 'gate_linear': '16/16', 'gate_rrf': '16/16'}


n=10000 w=0 selective  fields=none       hybrid_linear  qps=  4008.8 p50=0.25ms p99=0.34ms


...

n=500000 w=6 broad      fields=title+text hybrid_linear  qps=    14.3 p50=69.43ms p99=95.67ms


n=500000 w=6 broad      fields=title+text hybrid_rrf     qps=    14.2 p50=70.09ms p99=95.08ms


n=500000 w=6 broad      fields=title+text search_branch  qps=    19.3 p50=51.28ms p99=75.61ms


n=500000 w=6 broad      fields=title+text vsim_branch    qps=   110.9 p50=9.15ms p99=11.14ms


saved results.json / results.csv


## Gates

`gate_*` = FT.HYBRID ≡ oracle fusion (out of 16 queries), tie-aware.

In [3]:
pd.DataFrame(gates)

,size,selectivity,matches_mean,gate_linear,gate_rrf
0,10000,selective,2.800781,16/16,16/16
1,10000,medium,1055.027344,16/16,16/16
2,10000,broad,10000.000000,16/16,16/16
3,100000,selective,8.039062,16/16,16/16
4,100000,medium,9260.433594,16/16,16/16
5,100000,broad,100000.000000,16/16,16/16
6,500000,selective,14.582031,16/16,16/16
7,500000,medium,46688.316406,16/16,16/16
8,500000,broad,500000.000000,16/16,16/16


## Results — p50 latency (ms)

Rows = configuration (including fields mode), columns = contender. The ticket's answer is visible here: `hybrid_*` tracks `search_branch` (its own text top-K work) in every cell, and both track |matches|, not corpus size.

In [4]:
df = pd.DataFrame(results)
pivot = df.pivot_table(index=['size', 'workers', 'selectivity', 'fields'], columns='contender',
                       values='p50_ms', sort=False).round(2)
pivot[['hybrid_linear', 'hybrid_rrf', 'search_branch', 'vsim_branch']]

contender                              hybrid_linear  hybrid_rrf  \
size   workers selectivity fields                                  
10000  0       selective   none                 0.25        0.21   
                           title+text           0.23        0.26   
               medium      none                 1.31        1.35   
                           title+text           1.50        1.54   
               broad       none                13.46       13.90   
                           title+text          16.14       15.69   
       6       selective   none                 0.22        0.25   
                           title+text           0.25        0.25   
               medium      none                 1.00        1.01   
                           title+text           1.25        1.24   
               broad       none                 9.07        8.34   
                           title+text           9.70        9.79   
100000 0       selective   none                 0.35        0.27   
                           title+text           0.30        0.30   
               medium      none                 2.99        3.02   
                           title+text           3.15        3.11   
               broad       none                28.99       28.70   
                           title+text          31.42       31.42   
       6       selective   none                 0.34        0.36   
                           title+text           0.39        0.34   
               medium      none                 2.38        2.36   
                           title+text           2.47        2.55   
               broad       none                19.43       20.70   
                           title+text          21.07       21.75   
500000 0       selective   none                 0.34        0.35   
                           title+text           0.38        0.36   
               medium      none                10.65       10.09   
                           title+text          10.14        9.88   
               broad       none                79.06       81.66   
                           title+text          81.56       82.51   
       6       selective   none                 0.40        0.34   
                           title+text           0.43        0.37   
               medium      none                 9.78        9.17   
                           title+text           9.54        9.15   
               broad       none                67.88       67.74   
                           title+text          69.43       70.09   

contender                              search_branch  vsim_branch  
size   workers selectivity fields                                  
10000  0       selective   none                 0.06         0.15  
                           title+text           0.06         0.24  
               medium      none                 0.46         0.36  
                           title+text           0.55         0.43  
               broad       none                 2.53         6.07  
                           title+text           2.50         5.79  
       6       selective   none                 0.07         0.17  
                           title+text           0.09         0.24  
               medium      none                 0.54         0.41  
                           title+text           0.62         0.46  
               broad       none                 2.40         5.86  
                           title+text           2.83         6.36  
100000 0       selective   none                 0.06         0.21  
                           title+text           0.07         0.26  
               medium      none                 1.49         0.65  
                           title+text           1.56         0.71  
               broad       none                10.41         8.14  
                           title+text          11.29         8.47  
       6       selective   none                 0.07   

### ε — machinery overhead (fields=none only)

FT.HYBRID mean latency minus its **slowest branch** (the honest reference: branches can
deplete concurrently). Three components explain ε:

1. **Workers=0 depletes branches sequentially**, so the *other* branch's time is included.
2. **`YIELD_SCORE_AS` on the SEARCH subquery costs O(scanned docs)** — measured A/B at
   broad/100K with shallow windows (`debug_yield.py`): hybrid without any YIELD_SCORE_AS
   ≈ its branch (ε ≈ 2%); adding YIELD on SEARCH alone → +2ms. The score alias is written
   per scanned doc instead of only for the top-WINDOW survivors — **actionable
   optimization** (move the write post-sorter). YIELD on VSIM/COMBINE is free.
3. **Deep-window fusion in the merger** — with K/WINDOW = 1000/2000 the merger fuses
   ~3K unique docs per broad query (FT.PROFILE: `Hybrid Merger … Results processed: 2632`
   at 10K/broad): dict upkeep, per-doc merge, and a ~3K-row tail sort. This is the cost
   deliberately accepted to exercise the vector branch; it scales with WINDOW, not with
   corpus size.

In [5]:
m = df[df.fields == 'none'].pivot_table(index=['size', 'workers', 'selectivity'],
                   columns='contender', values='mean_ms', sort=False)
eps = (m['hybrid_linear'] - m[['search_branch', 'vsim_branch']].max(axis=1))
pd.DataFrame({'hybrid_mean_ms': m['hybrid_linear'].round(2),
              'slowest_branch_ms': m[['search_branch', 'vsim_branch']].max(axis=1).round(2),
              'eps_ms': eps.round(2),
              'eps_pct': (100 * eps / m['hybrid_linear']).round(1)})

hybrid_mean_ms  slowest_branch_ms  eps_ms  eps_pct
size   workers selectivity                                                    
10000  0       selective              0.25               0.15    0.10     39.5
               medium                 1.33               0.47    0.87     65.0
               broad                 13.62               6.10    7.52     55.2
       6       selective              0.23               0.17    0.06     25.3
               medium                 1.02               0.55    0.47     46.2
               broad                  9.18               5.99    3.19     34.8
100000 0       selective              0.35               0.21    0.14     40.5
               medium                 3.09               1.58    1.51     48.9
               broad                 29.69              11.17   18.52     62.4
       6       selective              0.34               0.20    0.14     40.3
               medium                 2.50               1.59    0.91     36.5
               broad                 20.22              11.35    8.86     43.8
500000 0       selective              0.35               0.25    0.10     29.5
               medium                11.26               8.28    2.98     26.4
               broad                 79.56              53.10   26.46     33.3
       6       selective              0.41               0.26    0.15     35.5
               medium                10.43               8.25    2.18     20.9
               broad                 68.38              51.72   16.66     24.4

## Notes / deviations from the plan

- N_TIMED=3000 per cell after 300 warm-up (plan said 30K; scaled for local run time —
  p99.9 still indicative).
- Embeddings: the dataset's precomputed 512-d vectors serve as both doc and query vectors
  (no re-embedding; engine behavior doesn't depend on which model produced them).
- Client timing includes redis-py round-trip (~50-100µs), identical for all contenders.
- Side finding 1: FT.AGGREGATE `ADDSCORES + SORTBY @__score MAX 20` measures ~30-45% slower
  than `FT.SEARCH … LIMIT 0 20` for the identical top-K-by-BM25 job (profiled: costlier
  Scorer + Sorter). Potential optimization target.
- Side finding 2: `YIELD_SCORE_AS` on the SEARCH subquery adds an O(scanned-docs) per-doc
  write (~2ms/query at broad/100K, ~25% of hybrid latency; `debug_yield.py`). PERF-473's
  command used it. Potential optimization: write the alias only for top-WINDOW survivors.
- `FT.PROFILE` captures for hybrid (workers=0, selective/broad per size) are saved in
  `results.json` under `profiles`.